# Example: a component with a known parameter

A detector measures a weak signal on top of noise. The noise is centered at zero by calibration, so its mean is
known; its spread, the signal and the share of the signal are not. Fixing the known mean with `family.view(mu=0)`
removes a degree of freedom that the data can hardly determine when the signal is small.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from pysatl_mpest.initializers import ClusterInitializer, KMeansPartitioner

## One sample

In [ ]:
truth = MixtureModel([normal(mu=0, sigma=1), normal(mu=2, sigma=0.5)], weights=[0.85, 0.15])
X = truth.generate(size=200, random_state=0)

free = [normal, normal]
fixed = [normal.view(mu=0), normal]
results = {}
for name, families in {"free mean": free, "noise mean fixed at 0": fixed}.items():
    start = ClusterInitializer(KMeansPartitioner()).initialize(X, families, random_state=0)
    results[name] = em(200).fit(X, start)
    print(name)
    show(results[name])

fig, ax = plt.subplots(figsize=(8, 3.5))
plot_mixtures(ax, X, {"truth": truth, **results}, bins=40)
plt.show()

The component with the fixed mean has only `sigma` among its free parameters: the mean is part of its family.

## Twenty samples

One sample can be lucky. Over twenty samples, the error of the signal — the distance of its mean to 2 plus the
distance of its weight to 0.15 — shows the effect of the fixed parameter.

In [ ]:
errors = {"free mean": [], "noise mean fixed at 0": []}
for seed in range(20):
    X = truth.generate(size=200, random_state=seed)
    for name, families in {"free mean": free, "noise mean fixed at 0": fixed}.items():
        fitted = em(100).fit(X, ClusterInitializer(KMeansPartitioner()).initialize(X, families, random_state=0))
        errors[name].append(abs(fitted[1].parameters["mu"] - 2) + abs(fitted.weights[1] - 0.15))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot(list(errors.values()), tick_labels=list(errors))
ax.set_ylabel("error of the signal")
plt.show()
for name, values in errors.items():
    print(f"{name:25} median error {np.median(values):.3f}")

assert np.median(errors["noise mean fixed at 0"]) < np.median(errors["free mean"])

## Takeaway

Fix what you know. The initializer and the estimators keep a fixed parameter exactly, and the order of the components
follows `families`, so the fixed component is always component 0.